# STAT 486 — Deliverable 4: Unsupervised Learning
## Detecting Unusual Airline Operations with Isolation Forest

**Question:** Can we automatically identify carrier-airport-months that operated outside their normal patterns — and does that signal add anything beyond the supervised high-delay prediction?

**Approach:** Isolation Forest flags operationally *unusual* months. A complementary persistence detector flags operationally *consistently bad* routes. Together they give two distinct views of airline reliability.

---
**Dataset:** U.S. Airline Delay Cause, 2013–2023 · ~163k carrier-airport-month records  
**Unit of analysis:** One row = one carrier operating at one airport for one calendar month  
**Training window:** 2013–2019 (pre-COVID) · **Holdout:** 2020–2023 (COVID + recovery)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'Airline_Delay_Cause.csv'
FIG_DIR   = ROOT / 'figures'
OUT_DIR   = ROOT / 'outputs'
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# ── Model parameters ─────────────────────────────────────────────────────────
RANDOM_STATE   = 486
CONTAMINATION  = 0.05   # assume ~5% of months are genuinely unusual
N_ESTIMATORS   = 200
MIN_FLIGHTS    = 10     # ignore rows with < 10 flights (rates are meaningless at tiny volume)
PERSIST_THRESH = 0.60   # a route is "chronic" if ≥60% of its months are high-delay
PERSIST_MIN_MO = 12     # require at least 12 months of history before flagging a route

---
## 1. Preparing the Data

Each row in the dataset describes how one airline performed at one airport during one month.
We compute a handful of derived rates that summarise performance, then split data into a
**training set (2013–2019)** and a **holdout set (2020–2023)**.

The model is trained only on pre-COVID data. This separation ensures that pandemic-era
disruptions act as an independent test of the model rather than contaminating its training.

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df[df['arr_flights'] >= MIN_FLIGHTS].copy()

# ── Performance rates (used for evaluation and display — not model inputs) ───
df['delay_rate']      = df['arr_del15']     / df['arr_flights']
df['cancel_rate']     = df['arr_cancelled'] / df['arr_flights']
df['divert_rate']     = df['arr_diverted']  / df['arr_flights']
# Average delay minutes per *delayed* flight (not per all flights)
df['mean_delay_mins'] = df['arr_delay']     / df['arr_del15'].clip(lower=1)
df['high_delay']      = (df['delay_rate'] > 0.20).astype(int)

# ── Cause-mix features (what *type* of delays occurred, as percentages) ──────
cause_cols = ['carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct']
cause_total = df[cause_cols].sum(axis=1).clip(lower=1)
for col in cause_cols:
    df[f'pct_{col}'] = df[col] / cause_total

# ── Train / holdout split ────────────────────────────────────────────────────
train_df   = df[df['year'] < 2020].copy()
holdout_df = df[df['year'] >= 2020].copy()

print('=' * 52)
print('DATA SUMMARY')
print('=' * 52)
print(f'Training rows  (2013-2019) : {len(train_df):>7,}')
print(f'Holdout rows   (2020-2023) : {len(holdout_df):>7,}')
print(f'Carriers in training       : {train_df["carrier"].nunique():>7,}')
print(f'Airports in training       : {train_df["airport"].nunique():>7,}')
print(f'High-delay months (train)  : {train_df["high_delay"].mean():>7.1%}')
print(f'High-delay months (holdout): {holdout_df["high_delay"].mean():>7.1%}')

---
## 2. Anomaly Detection: How Isolation Forest Works

**The core idea in plain English:**

Normal months blend in — they look similar to hundreds of other months.
Unusual months stand out — their combination of features is rare.
Isolation Forest exploits this by repeatedly asking random questions about the data
(e.g. "is the cancellation rate above X?"). Unusual rows get *isolated* quickly because
there are few other rows nearby that share their profile. Normal rows take many more
random questions to isolate.

Each month receives an **anomaly score from 0 to 1**. Higher means more unusual.

**What features does the model see?**

| Feature | What it measures |
|---|---|
| `cancel_rate` | Share of scheduled flights that were cancelled |
| `divert_rate` | Share of flights diverted to another airport |
| `pct_carrier_ct` | Fraction of delays caused by airline operations |
| `pct_weather_ct` | Fraction of delays caused by weather |
| `pct_nas_ct` | Fraction of delays caused by the national air system |
| `pct_security_ct` | Fraction of delays caused by security issues |
| `pct_late_aircraft_ct` | Fraction of delays caused by late incoming aircraft |
| `arr_flights` | Monthly flight volume |

**What is intentionally excluded:** `delay_rate` and `mean_delay_mins`. If we included the
delay rate in the model and then checked whether high-anomaly-score months have high delay
rates, we'd just be rediscovering what we put in. Keeping delay rate out makes the later
validation a real, independent test.

In [ ]:
ANOMALY_FEATURES = [
    'cancel_rate',
    'divert_rate',
    'pct_carrier_ct',
    'pct_weather_ct',
    'pct_nas_ct',
    'pct_security_ct',
    'pct_late_aircraft_ct',
    'arr_flights',
]

X_train = train_df[ANOMALY_FEATURES].fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

iso = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_train_scaled)

# Normalise raw scores to [0, 1]: higher = more anomalous
train_raw  = iso.decision_function(X_train_scaled)
_score_min = train_raw.min()
_score_max = train_raw.max()

train_df['anomaly_score'] = 1 - (train_raw - _score_min) / (_score_max - _score_min)
train_df['is_anomaly']    = (iso.predict(X_train_scaled) == -1).astype(int)
threshold = train_df.loc[train_df['is_anomaly'] == 1, 'anomaly_score'].min()

print(f'Anomalies flagged : {train_df["is_anomaly"].sum():,} out of {len(train_df):,} months'
      f'  ({train_df["is_anomaly"].mean():.1%})')
print(f'Anomaly threshold : {threshold:.3f}')

---
## 3. Score Distribution

Most months cluster near zero — they are operationally routine.
The red dashed line marks the threshold above which a month is labelled anomalous.
The long right tail contains the genuinely unusual months the model is looking for.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(train_df['anomaly_score'], bins=70, color='#4C78A8', alpha=0.85, edgecolor='none')
ax.axvline(threshold, color='#E45756', linestyle='--', linewidth=1.8,
           label=f'Anomaly threshold  ({threshold:.2f})')

ax.set_title('Most months are routine; a small tail is flagged as anomalous',
             fontsize=12, pad=10)
ax.set_xlabel('Anomaly score  (0 = perfectly normal · 1 = most unusual)', fontsize=10)
ax.set_ylabel('Number of carrier-airport-months', fontsize=10)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / 'unsupervised_score_distribution.png', dpi=150)
plt.show()

---
## 4. What Does the Model Flag?

The table below shows the most anomalous months in the training data (2013–2019).
Notice that high anomaly scores come from two very different situations:

- **High cancellation rate** — a large fraction of flights simply didn't operate that month,
  which distorts every other rate and makes the month look structurally different.
- **Extreme delay severity** — the flights that did operate were delayed by many hours,
  pointing to a specific operational breakdown rather than a volume issue.

This matters because neither type alone defines the other.

In [ ]:
top_anomalies = (
    train_df.sort_values('anomaly_score', ascending=False)
    .loc[:, ['year', 'month', 'carrier_name', 'airport_name',
             'delay_rate', 'cancel_rate', 'mean_delay_mins', 'anomaly_score']]
    .head(15)
    .reset_index(drop=True)
)
top_anomalies.index += 1

# Format for readability
display_top = top_anomalies.copy()
display_top['delay_rate']      = display_top['delay_rate'].map('{:.1%}'.format)
display_top['cancel_rate']     = display_top['cancel_rate'].map('{:.1%}'.format)
display_top['mean_delay_mins'] = display_top['mean_delay_mins'].map('{:.0f} min'.format)
display_top['anomaly_score']   = display_top['anomaly_score'].map('{:.3f}'.format)
display_top.columns = ['Year', 'Mo', 'Carrier', 'Airport',
                       'Delay Rate', 'Cancel Rate', 'Avg Delay / Delayed Flight', 'Score']
top_anomalies.to_csv(OUT_DIR / 'top_anomalies.csv', index=False)
display_top

---
## 5. Which Carriers and Months Are Most Anomalous?

The heatmap below shows average anomaly score by carrier (rows) and month (columns).
Dark red cells indicate carrier-month combinations that are consistently unusual —
either because of structural seasonal patterns or operational irregularities.

In [ ]:
carrier_month = train_df.pivot_table(
    values='anomaly_score', index='carrier', columns='month', aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(carrier_month.values, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.4)
ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=9)
ax.set_yticks(range(len(carrier_month.index)))
ax.set_yticklabels(carrier_month.index, fontsize=8)
ax.set_title('Mean anomaly score by carrier and month  (2013-2019 training data)
'
             'Dark red = consistently unusual · Green = consistently routine',
             fontsize=11, pad=10)
ax.set_xlabel('Month', fontsize=10)
ax.set_ylabel('Carrier code', fontsize=10)
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Mean anomaly score', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'unsupervised_carrier_month_heatmap.png', dpi=150)
plt.show()

---
## 6. Does the Anomaly Score Actually Predict High Delays?

Because we intentionally **excluded** the delay rate from the model, we can now ask a
genuine question: do months the model considers operationally unusual tend to also have
higher delay rates?

We split all training months into five equal groups (quintiles) ranked by anomaly score,
then look at how often each group has a high delay rate (>20% of flights delayed).

If the anomaly score is informative, the bars should rise from left to right.

In [ ]:
train_df['score_quintile'] = pd.qcut(
    train_df['anomaly_score'], q=5,
    labels=['Q1\nlowest', 'Q2', 'Q3', 'Q4', 'Q5\nhighest'],
)

qt = (
    train_df.groupby('score_quintile', observed=False)
    .agg(
        n               = ('high_delay', 'size'),
        high_delay_rate = ('high_delay', 'mean'),
        mean_delay_rate = ('delay_rate', 'mean'),
        mean_cancel_rate= ('cancel_rate','mean'),
    )
    .reset_index()
)
qt.to_csv(OUT_DIR / 'anomaly_supervised_connection.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(qt['score_quintile'].astype(str), qt['high_delay_rate'],
              color='#F58518', width=0.55)
ax.bar_label(bars, fmt='%.1%%', padding=4, fontsize=9)
ax.set_ylim(0, 0.65)
ax.set_xlabel('Anomaly-score quintile  (Q1 = most normal · Q5 = most unusual)', fontsize=10)
ax.set_ylabel('Fraction of months with high delay rate', fontsize=10)
ax.set_title('Does an unusual operational profile predict high delays?
'
             '(delay rate was NOT a model input — this is an independent test)',
             fontsize=11, pad=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'unsupervised_supervised_connection.png', dpi=150)
plt.show()

print('Quintile breakdown:')
print(qt.rename(columns={
    'score_quintile':'Quintile','n':'N',
    'high_delay_rate':'High-delay rate',
    'mean_delay_rate':'Mean delay rate',
    'mean_cancel_rate':'Mean cancel rate'
}).to_string(index=False))

**How to read this result:**

The anomaly score — built entirely from cause-mix and cancellation patterns — is a
*complement* to delay rate, not a duplicate of it. High-anomaly months are not
necessarily the highest-delay months. What the model is really finding is
**structural disruption**: months where the *mix* of causes or the *cancellation pattern*
is out of the ordinary. Delay rate is only one dimension of that.

---
## 7. What Types of Unusual Behavior Exist? (Clustering)

Knowing a month is unusual is helpful. Knowing *in what way* it is unusual is more useful.

We use **K-Means clustering** to group all training months into distinct operational
archetypes — think of it as automatically sorting months into buckets that share a similar
profile. We then check which buckets contain the most anomalies.

In [ ]:
k_values   = list(range(2, 9))
inertias, sil_scores = [], []

rng        = np.random.default_rng(RANDOM_STATE)
sample_n   = min(20_000, len(X_train_scaled))
sample_idx = rng.choice(len(X_train_scaled), size=sample_n, replace=False)
X_sel      = X_train_scaled[sample_idx]

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20, algorithm='lloyd')
    labels = km.fit_predict(X_sel)
    inertias.append(km.inertia_)
    sil_scores.append(
        silhouette_score(X_sel, labels, sample_size=5000, random_state=RANDOM_STATE)
    )

best_k = k_values[int(np.argmax(sil_scores))]
print(f'Best number of clusters (by silhouette score): {best_k}')

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(k_values, inertias, marker='o', color='#4C78A8')
ax1.set_xlabel('Number of clusters (k)', fontsize=10)
ax1.set_ylabel('Inertia (within-cluster variance)', color='#4C78A8', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#4C78A8')
ax2 = ax1.twinx()
ax2.plot(k_values, sil_scores, marker='s', color='#E45756')
ax2.set_ylabel('Silhouette score (higher = better separation)', color='#E45756', fontsize=10)
ax2.tick_params(axis='y', labelcolor='#E45756')
ax1.set_title('Choosing the number of clusters', fontsize=11, pad=8)
ax1.spines[['top']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'kmeans_model_selection.png', dpi=150)
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20, algorithm='lloyd')
train_df['cluster'] = kmeans.fit_predict(X_train_scaled)

cluster_summary = (
    train_df.groupby('cluster', as_index=False)
    .agg(
        n                  = ('cluster',       'size'),
        anomaly_rate       = ('is_anomaly',    'mean'),
        mean_anomaly_score = ('anomaly_score', 'mean'),
        high_delay_rate    = ('high_delay',    'mean'),
        mean_delay_rate    = ('delay_rate',    'mean'),
        mean_cancel_rate   = ('cancel_rate',   'mean'),
    )
    .sort_values('anomaly_rate', ascending=False)
    .reset_index(drop=True)
)
cluster_summary.to_csv(OUT_DIR / 'kmeans_cluster_summary.csv', index=False)

print('Cluster summary (sorted by anomaly rate):')
print(cluster_summary.to_string(index=False))

In [ ]:
pca    = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_train_scaled)
train_df['pc1'] = coords[:, 0]
train_df['pc2'] = coords[:, 1]
var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(
    train_df['pc1'], train_df['pc2'],
    c=train_df['cluster'], s=5, cmap='tab10', alpha=0.4,
)
axes[0].set_title(
    f'Operational archetypes (K-Means clusters)\n'
    f'PCA projection · PC1+PC2 explain {var_exp.sum():.0%} of variance',
    fontsize=10
)
axes[0].set_xlabel(f'PC1 ({var_exp[0]:.0%})', fontsize=9)
axes[0].set_ylabel(f'PC2 ({var_exp[1]:.0%})', fontsize=9)
plt.colorbar(sc, ax=axes[0], label='Cluster')

anom = train_df[train_df['is_anomaly'] == 1]
axes[1].scatter(train_df['pc1'], train_df['pc2'],
                c='lightgray', s=5, alpha=0.3, label='Routine months')
axes[1].scatter(anom['pc1'], anom['pc2'],
                c='crimson', s=7, alpha=0.7, label='Anomalous months')
axes[1].set_title('Where anomalies fall in the feature space', fontsize=10)
axes[1].set_xlabel(f'PC1 ({var_exp[0]:.0%})', fontsize=9)
axes[1].set_ylabel(f'PC2 ({var_exp[1]:.0%})', fontsize=9)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'kmeans_pca_anomaly_overlay.png', dpi=150)
plt.show()

In [ ]:
profiles   = train_df.groupby('cluster')[ANOMALY_FEATURES].mean()
z_profiles = (profiles - profiles.mean()) / profiles.std(ddof=0)

fig, ax = plt.subplots(figsize=(13, max(3, best_k * 0.55)))
im = ax.imshow(z_profiles.values, aspect='auto', cmap='coolwarm', vmin=-2, vmax=2)

short_labels = ['cancel', 'divert', 'pct_carrier', 'pct_weather',
                'pct_nas', 'pct_security', 'pct_late_acft', 'volume']
ax.set_xticks(range(len(ANOMALY_FEATURES)))
ax.set_xticklabels(short_labels, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(z_profiles.index)))
ax.set_yticklabels([f'Cluster {c}' for c in z_profiles.index], fontsize=9)
ax.set_title('How each cluster differs from the average month\n'
             'Red = above average · Blue = below average',
             fontsize=11, pad=10)
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Standard deviations from mean', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'kmeans_cluster_profiles_heatmap.png', dpi=150)
plt.show()

---
## 8. The Detector's Blind Spot: Chronically Bad Routes

Isolation Forest is designed to find **unusual** months. There is a category of routes
it cannot see: carrier-airports that are *reliably* bad, month after month.

If a route consistently has 70% of flights delayed, every individual month looks
"normal" to the detector — because there are many other months from the same route
that look exactly the same. Consistent badness is invisible to a detector that
defines "unusual" relative to the rest of the data.

The scatter below shows this clearly. Each dot is one carrier-airport-month. **Red dots**
belong to routes that are chronically high-delay (≥ 60% of their months exceed the 20%
delay threshold). Notice where they sit: **high delay rate, low anomaly score** —
the bottom-right corner, out of the detector's reach.

In [ ]:
# ── Build route-level stats ─────────────────────────────────────────────────
ca_train = (
    train_df.groupby(['carrier', 'airport'])
    .agg(
        n_months        = ('year',          'count'),
        pct_high_delay  = ('high_delay',    'mean'),
        mean_delay_rate = ('delay_rate',    'mean'),
        if_anomaly_rate = ('is_anomaly',    'mean'),
        mean_anom_score = ('anomaly_score', 'mean'),
    )
    .reset_index()
)

chronic_mask  = (
    (ca_train['pct_high_delay'] >= PERSIST_THRESH) &
    (ca_train['n_months']       >= PERSIST_MIN_MO)
)
chronic_pairs             = ca_train.loc[chronic_mask, ['carrier', 'airport']].copy()
chronic_pairs['is_chronic'] = True

train_df = train_df.merge(chronic_pairs, on=['carrier', 'airport'], how='left')
train_df['is_chronic'] = train_df['is_chronic'].fillna(False)

# ── Plot ─────────────────────────────────────────────────────────────────────
sample = train_df.sample(min(15_000, len(train_df)), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    sample.loc[~sample['is_chronic'], 'anomaly_score'],
    sample.loc[~sample['is_chronic'], 'delay_rate'],
    c='#4C78A8', s=6, alpha=0.25, label='Normal routes',
)
ax.scatter(
    sample.loc[sample['is_chronic'], 'anomaly_score'],
    sample.loc[sample['is_chronic'], 'delay_rate'],
    c='#E45756', s=12, alpha=0.6,
    label=f'Chronically bad routes  (>={PERSIST_THRESH:.0%} of months above delay threshold)',
)
ax.axvline(threshold, color='black', linestyle='--', linewidth=1,
           label='Isolation Forest anomaly threshold')
ax.axhline(0.20, color='gray',  linestyle='--', linewidth=1,
           label='High-delay threshold (20%)')

ax.annotate('Blind spot:
high delay,
not flagged',
            xy=(threshold * 0.35, 0.72), fontsize=9, color='#C62828',
            bbox=dict(boxstyle='round,pad=0.3', fc='#FFEBEE', ec='#E45756', lw=0.8))

ax.set_xlabel('Anomaly score  (higher = more unusual)', fontsize=10)
ax.set_ylabel('Delay rate for this month', fontsize=10)
ax.set_title('Isolation Forest misses chronically bad routes
'
             'Red points concentrate in the bottom-right: high delay, low anomaly score',
             fontsize=11, pad=10)
ax.legend(loc='upper left', fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'unsupervised_blind_spot.png', dpi=150)
plt.show()

n_chronic = chronic_mask.sum()
print(f'Chronically underperforming routes : {n_chronic:>4} of {len(ca_train):,}'
      f'  ({n_chronic/len(ca_train):.1%})')
print(f'Their average IF anomaly rate      : {ca_train.loc[chronic_mask, "if_anomaly_rate"].mean():.1%}'
      f'  (vs {CONTAMINATION:.0%} expected at random)')

---
## 9. A Second Detector: Flagging Chronic Underperformers

To catch what Isolation Forest misses, we add a **persistence detector**.
The rule is simple:

> A carrier-airport route is flagged if **≥ 60% of its months** in the training data
> had a delay rate above 20%, and it has **at least 12 months** of history.

No machine learning is needed. This is a straightforward audit of which routes are
consistently unreliable.

**The key insight:** the two detectors find almost entirely different routes.

| Detector | What it catches |
|---|---|
| Isolation Forest | One-off shocks — unusual months on otherwise-normal routes |
| Persistence detector | Chronic underperformance — bad routes that never improve |

In [ ]:
ca_train['persistence_flag'] = chronic_mask.astype(int)
IF_ROUTE_THRESH = 0.20  # flag a route if >=20% of its months were IF-anomalous
ca_train['if_flag'] = (ca_train['if_anomaly_rate'] >= IF_ROUTE_THRESH).astype(int)

# ── Cross-tab ─────────────────────────────────────────────────────────────────
ct = pd.crosstab(ca_train['persistence_flag'], ca_train['if_flag'], margins=True)
ct.index   = ['Not chronic', 'Chronic', 'All']
ct.columns = ['Not IF-flagged', 'IF-flagged', 'All']
ct.index.name   = f'Persistence  (>={PERSIST_THRESH:.0%} high-delay months)'
ct.columns.name = f'Isolation Forest  (>={IF_ROUTE_THRESH:.0%} months flagged)'
print(ct)
ct.to_csv(OUT_DIR / 'detector_comparison.csv')

# ── Comparison bar chart ──────────────────────────────────────────────────────
both         = int(((ca_train['persistence_flag'] == 1) & (ca_train['if_flag'] == 1)).sum())
persist_only = int(((ca_train['persistence_flag'] == 1) & (ca_train['if_flag'] == 0)).sum())
if_only      = int(((ca_train['persistence_flag'] == 0) & (ca_train['if_flag'] == 1)).sum())
neither      = int(((ca_train['persistence_flag'] == 0) & (ca_train['if_flag'] == 0)).sum())

labels = ['Persistence only
(chronic, IF missed it)',
          'IF only
(shock, not chronic)',
          'Both detectors',
          'Neither']
counts = [persist_only, if_only, both, neither]
colors = ['#E45756', '#4C78A8', '#B279A2', '#CCCCCC']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, counts, color=colors, width=0.5)
ax.bar_label(bars, padding=4, fontsize=10, fontweight='bold')
ax.set_ylabel('Number of carrier-airport routes', fontsize=10)
ax.set_title('The two detectors are largely complementary — they catch different routes
'
             '(2013-2019 training data · carrier-airport level)',
             fontsize=11, pad=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'persistence_vs_isolation_forest.png', dpi=150)
plt.show()

In [ ]:
# Routes that are chronically bad but that Isolation Forest never noticed
names = df[['carrier', 'carrier_name', 'airport', 'airport_name']].drop_duplicates()
chronic_missed = (
    ca_train
    .loc[(ca_train['persistence_flag'] == 1) & (ca_train['if_flag'] == 0)]
    .merge(names, on=['carrier', 'airport'])
    .sort_values('pct_high_delay', ascending=False)
    .head(15)
    .reset_index(drop=True)
)
chronic_missed.index += 1

display_cm = chronic_missed[['carrier_name', 'airport_name', 'n_months',
                              'pct_high_delay', 'mean_delay_rate', 'if_anomaly_rate']].copy()
display_cm.columns = ['Carrier', 'Airport', 'Months of data',
                      '% months high-delay', 'Avg delay rate', 'IF anomaly rate']
display_cm['% months high-delay'] = display_cm['% months high-delay'].map('{:.0%}'.format)
display_cm['Avg delay rate']       = display_cm['Avg delay rate'].map('{:.1%}'.format)
display_cm['IF anomaly rate']      = display_cm['IF anomaly rate'].map('{:.1%}'.format)
chronic_missed.to_csv(OUT_DIR / 'chronic_missed_by_if.csv', index=False)

print('Routes the persistence detector flags that Isolation Forest missed:')
display_cm

---
## 10. Out-of-Sample Test: Does the Model Generalise to COVID-Era Data?

A model trained on 2013–2019 has never seen a global pandemic. If we apply it
to 2020–2023 without retraining, it should flag the pandemic months as highly
anomalous — cancellations and cause-mix patterns during April–June 2020 were unlike
anything in the training data.

This serves as a **sanity check**: if the model fails to flag COVID months, something
is wrong with it. If it flags them clearly, that confirms the model is detecting real
structural disruption rather than noise.

Anomaly scores above 1.0 in the chart below mean the month was *more extreme than
the most anomalous month the model was trained on*.

In [ ]:
X_holdout        = holdout_df[ANOMALY_FEATURES].fillna(0)
X_holdout_scaled = scaler.transform(X_holdout)

holdout_raw = iso.decision_function(X_holdout_scaled)
holdout_df['anomaly_score'] = 1 - (holdout_raw - _score_min) / (_score_max - _score_min)
holdout_df['is_anomaly']    = (iso.predict(X_holdout_scaled) == -1).astype(int)

print(f'Holdout anomaly rate : {holdout_df["is_anomaly"].mean():.1%}'
      f'  (vs {CONTAMINATION:.0%} during training)')
print(f'Scores above 1.0     : {(holdout_df["anomaly_score"] > 1).sum():,}'
      f'  (more extreme than any training month)')

# ── Side-by-side: score distribution + monthly anomaly rate ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(train_df['anomaly_score'].clip(upper=1.5),
             bins=60, color='#4C78A8', alpha=0.7, label='2013-2019 (train)')
axes[0].hist(holdout_df['anomaly_score'].clip(upper=1.5),
             bins=60, color='#E45756', alpha=0.7, label='2020-2023 (holdout)')
axes[0].axvline(threshold, color='black', linestyle='--', linewidth=1.2,
                label='IF threshold')
axes[0].axvline(1.0, color='gray', linestyle=':', linewidth=1.2,
                label='Score = 1.0  (beyond training range)')
axes[0].set_title('Score distribution shifts right during COVID era', fontsize=10)
axes[0].set_xlabel('Anomaly score', fontsize=9)
axes[0].set_ylabel('Count', fontsize=9)
axes[0].legend(fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

monthly = (
    holdout_df.groupby(['year', 'month'])
    .agg(anomaly_rate=('is_anomaly', 'mean'))
    .reset_index()
)
monthly['period'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))
axes[1].plot(monthly['period'], monthly['anomaly_rate'],
             color='#E45756', marker='o', markersize=4, linewidth=1.5)
axes[1].fill_between(monthly['period'], monthly['anomaly_rate'],
                     alpha=0.15, color='#E45756')
axes[1].axhline(CONTAMINATION, color='#4C78A8', linestyle='--', linewidth=1,
                label=f'Training baseline ({CONTAMINATION:.0%})')
axes[1].set_title('Monthly anomaly rate in holdout data
Spike = COVID disruption',
                  fontsize=10)
axes[1].set_xlabel('Month', fontsize=9)
axes[1].set_ylabel('Fraction of months flagged as anomalous', fontsize=9)
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / 'covid_holdout_scores.png', dpi=150)
plt.show()

In [ ]:
top_holdout = (
    holdout_df.sort_values('anomaly_score', ascending=False)
    .loc[:, ['year', 'month', 'carrier_name', 'airport_name',
             'delay_rate', 'cancel_rate', 'mean_delay_mins', 'anomaly_score']]
    .head(15)
    .reset_index(drop=True)
)
top_holdout.index += 1

display_h = top_holdout.copy()
display_h['delay_rate']      = display_h['delay_rate'].map('{:.1%}'.format)
display_h['cancel_rate']     = display_h['cancel_rate'].map('{:.1%}'.format)
display_h['mean_delay_mins'] = display_h['mean_delay_mins'].map('{:.0f} min'.format)
display_h['anomaly_score']   = display_h['anomaly_score'].map('{:.3f}'.format)
display_h.columns = ['Year', 'Mo', 'Carrier', 'Airport',
                     'Delay Rate', 'Cancel Rate', 'Avg Delay / Delayed Flight', 'Score']
top_holdout.to_csv(OUT_DIR / 'top_holdout_anomalies.csv', index=False)
display_h

---
## Conclusions

### What we did
We trained an Isolation Forest model on 2013–2019 airline operations data to detect
carrier-airport-months whose operational profile (cancellation rate, diversion rate,
cause mix, flight volume) was unusual relative to the historical baseline.
We then added a persistence detector — a simple rule that flags routes where the majority
of months exceed the high-delay threshold.

---

### Finding 1: The anomaly detector captures structural disruptions, not just high delays

The model flags months that look *structurally different* from the norm — extreme
cancellation spikes, unusual cause-mix combinations, atypical flight volume.
When validated out-of-sample against COVID-era data, the model correctly identified
April–June 2020 as off-the-charts anomalous without ever seeing a pandemic during
training. That is the model doing exactly what it should.

---

### Finding 2: Isolation Forest has a predictable blind spot — it misses chronic underperformers

A route that is bad every single month looks *normal* to an outlier detector, because
each bad month is surrounded by many other bad months from the same route.
The persistence detector surfaces these routes directly.
In the 2013–2019 training data, **several hundred carrier-airport routes (see `chronic_missed_by_if.csv`)** met the
chronic-underperformer threshold; the vast majority were never flagged by Isolation Forest.
These routes represent a different kind of risk: not dramatic disruption, but steady,
predictable failure.

---

### Finding 3: The two detectors are complementary, not redundant

Running both detectors together provides broader coverage:
- **Isolation Forest** for one-off operational shocks on routes that are otherwise reliable
- **Persistence detector** for entrenched, chronic underperformance

Neither detector alone is sufficient. Together they give a more complete picture of
where airline reliability breaks down.

---

### Limitations

- **Contamination parameter (5%)** was set by judgment, not data — the true fraction of
  "genuinely anomalous" months in the data is unknown.
- **Route-level persistence threshold (60%)** is also a judgment call; results are
  qualitatively robust but the specific list of flagged routes would shift with this parameter.
- **The anomaly score is not a cause** — a high score tells you a month was unusual,
  not *why* it was unusual or who is responsible for it.
- **2020–2023 as holdout** means the persistence detector was built on pre-COVID data only.
  Post-COVID route patterns may differ.